In [1]:
#Cargar Montone Convex Soberanos
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CMT_TENORS = [6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
MIN_BONDS = 3

def mc_node_forwards(t, f_disc):
    n = len(t)
    f = np.zeros(n)
    for i in range(1, n - 1):
        h_l = t[i] - t[i - 1]
        h_r = t[i + 1] - t[i]
        f[i] = (f_disc[i - 1] * h_r + f_disc[i] * h_l) / (h_l + h_r)
    f[0] = f_disc[0] - 0.5 * (f[1] - f_disc[0])
    f[-1] = f_disc[-1] - 0.5 * (f[-2] - f_disc[-1])
    return np.maximum(f, 1e-6)

def mc_correct_fwd(f0, fd, f1):
    etas = np.linspace(0.01, 0.99, 300)
    f_cur = (
    f0 * (1 - 4 * etas + 3 * etas**2)
        + fd * 6 * etas * (1 - etas)
        + f1 * etas * (-2 + 3 * etas)
    )
    if f_cur.min() >= 0:
        return f0, fd, f1
    g = (
        f0 * (1 - 4 * etas + 3 * etas**2)
        + f1 * etas * (-2 + 3 * etas)
    )
    w = 6 * etas * (1 - etas)
    mask = w > 1e-12
    fd_new = max(fd, np.max(-g[mask] / w[mask]) + 1e-8)
    return max(f0, 1e-6), fd_new, max(f1, 1e-6)

def mc_yield_at(t_query, t, y_dec, Q, f_disc, f_node):
    hit = np.where(np.abs(t - t_query) < 1e-9)[0]
    if len(hit):
        return float(y_dec[hit[0]] * 100)
    if t_query < t[0] or t_query > t[-1]:
        return np.nan
    i = int(np.clip(int(np.searchsorted(t, t_query, side="right")) - 1, 0, len(t) - 2))
    t0 = t[i]
    t1 = t[i + 1]
    h = t1 - t0
    eta = (t_query - t0) / h
    f0, fd, f1 = mc_correct_fwd(f_node[i], f_disc[i], f_node[i + 1])
    Q_t = Q[i] + h * (
        f0 * (eta - 2 * eta**2 + eta**3)
        + fd * (3 * eta**2 - 2 * eta**3)
        + f1 * (-eta**2 + eta**3)
    )
    return float((Q_t / t_query) * 100)

def build_cmt(df_clean, tenors=CMT_TENORS, min_bonds=MIN_BONDS):
    records = []
    skipped = 0
    first_error = None
    df_work = df_clean.copy()
    df_work["Fecha"] = pd.to_datetime(df_work["Fecha"])
    for date, grp in df_work.groupby("Fecha"):
        grp = grp.dropna(subset=["Maturity", "FV.Rate"]).sort_values("Maturity").drop_duplicates("Maturity")
        if len(grp) < min_bonds:
            skipped += 1
            continue
        t_raw = grp["Maturity"].values
        y_raw = grp["FV.Rate"].values
        t_min = t_raw.min()
        t_max = t_raw.max()
        try:
            idx = np.argsort(t_raw)
            t_s = t_raw[idx]
            y_dec = y_raw[idx] / 100.0
            Q = y_dec * t_s
            f_disc = np.diff(Q) / np.diff(t_s)
            f_node = mc_node_forwards(t_s, f_disc)
            row = {"Fecha": date}
            for tau in tenors:
                if t_min <= tau <= t_max:
                    row[f"{tau}Y"] = round(mc_yield_at(tau, t_s, y_dec, Q, f_disc, f_node), 4)
                else:
                    row[f"{tau}Y"] = np.nan
            records.append(row)
        except Exception as e:
            if first_error is None:
                first_error = str(e)
            skipped += 1
    if len(records) == 0:
        print(f"ERROR: 0 records built. skipped={skipped}. First error: {first_error}")
        return pd.DataFrame()
    df_cmt = pd.DataFrame(records).sort_values("Fecha").reset_index(drop=True)
    df_cmt["Fecha"] = pd.to_datetime(df_cmt["Fecha"])
    print(f"CMT built: {len(df_cmt):,} dates | skipped: {skipped:,}")
    return df_cmt



print("functions loaded OK")
print("test:", mc_yield_at(8, np.array([6.0,10.0,15.0]), np.array([0.07,0.075,0.078]), np.array([0.07,0.075,0.078])*np.array([6.0,10.0,15.0]), np.diff(np.array([0.07,0.075,0.078])*np.array([6.0,10.0,15.0]))/np.diff(np.array([6.0,10.0,15.0])), mc_node_forwards(np.array([6.0,10.0,15.0]), np.diff(np.array([0.07,0.075,0.078])*np.array([6.0,10.0,15.0]))/np.diff(np.array([6.0,10.0,15.0])))))

functions loaded OK
test: 7.30625


In [2]:
# =============================================================================
# CELL 1 – DATA LOADING & CMT CONSTRUCTION
# =============================================================================
# Toggle USE_DUMMY to switch between dummy CSV data (for Claude Code / GitHub)
# and the real data pipeline on your work computer (R:/ drive).
# =============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# ── CONFIGURATION ────────────────────────────────────────────────────────────
USE_DUMMY = True  # Set to False on work computer to use real data from R:/

# Target CMT tenors for Soberanos
SOB_CMT_TENORS = [6, 7, 8, 9, 10, 11, 12, 13, 14, 15]

# Target CMT tenors for UST (to match Soberano nodes for LC Spread calculation)
UST_CMT_TENORS = [6, 7, 8, 9, 10, 11, 12, 13, 14, 15]

# UST raw nodes available in the CSV
UST_RAW_NODES = [1, 2, 3, 5, 7, 10, 20, 30]

# ── 1A. LOAD UST CMT ────────────────────────────────────────────────────────
if USE_DUMMY:
    PATH_UST = "USTs90s26s.csv"
else:
    PATH_UST = r"R:/RScripts/PCA Tools Frodo/PCA Tools Frodo/USTs90s26s.csv"

df_ust_raw = pd.read_csv(PATH_UST)

# Standardize column names: "1 yr" → "1Y", "date" → "Fecha"
col_map = {"date": "Fecha"}
for c in df_ust_raw.columns:
    c_strip = c.strip()
    if c_strip == "date":
        col_map[c] = "Fecha"
    elif "yr" in c_strip.lower():
        tenor = c_strip.split()[0]
        col_map[c] = f"{tenor}Y"
    else:
        col_map[c] = c_strip
df_ust_raw.rename(columns=col_map, inplace=True)

df_ust_raw["Fecha"] = pd.to_datetime(df_ust_raw["Fecha"], format="%m/%d/%Y")
df_ust_raw = df_ust_raw.sort_values("Fecha").reset_index(drop=True)
df_ust_raw = df_ust_raw.set_index("Fecha")

print(f"UST raw loaded: {df_ust_raw.index.min().date()} to {df_ust_raw.index.max().date()}")
print(f"  Shape: {df_ust_raw.shape}  |  Columns: {list(df_ust_raw.columns)}")

# ── 1B. MC-INTERPOLATE UST TO 6-15Y NODES ───────────────────────────────────
def mc_interpolate_wide(df_wide, input_tenors, output_tenors):
    """Apply monotone convex interpolation to a wide-format yield DataFrame.
    
    Parameters
    ----------
    df_wide : DataFrame with numeric tenor columns (yields in %) and DatetimeIndex.
    input_tenors : list of float, tenors corresponding to df_wide columns.
    output_tenors : list of int/float, desired output tenors.
    
    Returns
    -------
    DataFrame with Fecha index and columns like '6Y', '7Y', etc.
    """
    records = []
    for date, row in df_wide.iterrows():
        vals = row.values.astype(float)
        mask = ~np.isnan(vals)
        if mask.sum() < 3:
            continue
        t_s = np.array(input_tenors, dtype=float)[mask]
        y_pct = vals[mask]
        y_dec = y_pct / 100.0
        Q = y_dec * t_s
        f_disc = np.diff(Q) / np.diff(t_s)
        f_node = mc_node_forwards(t_s, f_disc)
        
        rec = {}
        for tau in output_tenors:
            if t_s.min() <= tau <= t_s.max():
                rec[f"{tau}Y"] = round(mc_yield_at(tau, t_s, y_dec, Q, f_disc, f_node), 4)
            else:
                rec[f"{tau}Y"] = np.nan
        records.append((date, rec))
    
    if not records:
        return pd.DataFrame()
    
    dates, rows = zip(*records)
    result = pd.DataFrame(list(rows), index=pd.DatetimeIndex(dates, name="Fecha"))
    return result

# Prepare UST for interpolation: extract numeric tenor columns
ust_tenor_cols = [c for c in df_ust_raw.columns if c.endswith("Y")]
ust_input_tenors = [int(c.replace("Y", "")) for c in ust_tenor_cols]

# df_ust_cmt: UST yields interpolated to 6-15Y (matches Soberano nodes)
df_ust_cmt = mc_interpolate_wide(
    df_ust_raw[ust_tenor_cols], ust_input_tenors, UST_CMT_TENORS
)
df_ust_cmt = df_ust_cmt.dropna()
print(f"\nUST CMT (MC-interpolated 6-15Y): {len(df_ust_cmt):,} dates")
print(f"  Columns: {list(df_ust_cmt.columns)}")

# ── 1C. UST YIELDS & SPREADS (original nodes for regime detection) ───────────
# Keep original UST raw for standard spread regime detection (2s10s etc.)
df_ust = df_ust_raw.copy()
ust_yields_bps = df_ust * 100

# Standard UST curve spreads (bps)
ust_spreads = pd.DataFrame(index=df_ust.index)
ust_spreads["2s5s"]   = (df_ust["5Y"]  - df_ust["2Y"])  * 100
ust_spreads["2s10s"]  = (df_ust["10Y"] - df_ust["2Y"])  * 100
ust_spreads["2s30s"]  = (df_ust["30Y"] - df_ust["2Y"])  * 100
ust_spreads["5s10s"]  = (df_ust["10Y"] - df_ust["5Y"])  * 100
ust_spreads["5s30s"]  = (df_ust["30Y"] - df_ust["5Y"])  * 100
ust_spreads["10s30s"] = (df_ust["30Y"] - df_ust["10Y"]) * 100

print(f"\nUST spreads computed: {list(ust_spreads.columns)}")

# ── 1D. LOAD & BUILD SOBERANOS CMT ──────────────────────────────────────────
if USE_DUMMY:
    # DUMMY PATH: dfsobsprecmt.csv is wide-format (date, 6Y, 7Y, 10Y, 20Y)
    # Needs MC interpolation to fill 8Y, 9Y, 11Y, 12Y, 13Y, 14Y, 15Y
    PATH_SOB = "dfsobsprecmt.csv"
    df_sob_raw = pd.read_csv(PATH_SOB)
    df_sob_raw.rename(columns={"date": "Fecha"}, inplace=True)
    df_sob_raw["Fecha"] = pd.to_datetime(df_sob_raw["Fecha"], format="%m/%d/%Y")
    df_sob_raw = df_sob_raw.sort_values("Fecha").set_index("Fecha")
    
    # Identify available tenor columns and their numeric values
    sob_raw_cols = [c for c in df_sob_raw.columns if c.endswith("Y")]
    sob_raw_tenors = [int(c.replace("Y", "")) for c in sob_raw_cols]
    
    df_sob_cmt = mc_interpolate_wide(
        df_sob_raw[sob_raw_cols], sob_raw_tenors, SOB_CMT_TENORS
    )
    df_sob_cmt = df_sob_cmt.dropna()
    print(f"\nSoberanos CMT (dummy, MC-interpolated): {len(df_sob_cmt):,} dates")
    
else:
    # REAL DATA PATH: load from R:/ drive (Bid/Offer closing runs → build_cmt)
    PATH_SOB_REAL = r"R:/RScripts/PCA Tools Frodo/PCA Tools Frodo/dfsobs.csv"
    df_sob_real = pd.read_csv(PATH_SOB_REAL, parse_dates=["Fecha"])
    df_sob_real["Fecha"] = pd.to_datetime(
        df_sob_real["Fecha"], errors="coerce", format="%d/%m/%Y"
    )
    df_clean = df_sob_real.dropna(subset=["Bid", "Offer"], how="all").copy()
    df_clean["FV.Rate"] = np.where(
        (df_clean["Bid"] <= 0) | pd.isna(df_clean["Bid"]),
        df_clean["Offer"] + 0.01,
        np.where(
            (df_clean["Offer"] <= 0) | pd.isna(df_clean["Offer"]),
            df_clean["Bid"] - 0.01,
            (df_clean["Bid"] + df_clean["Offer"]) / 2,
        ),
    )
    df_clean["Fecha"] = pd.to_datetime(
        df_clean["Fecha"], errors="coerce", format="%d/%m/%Y"
    )
    df_clean = df_clean[df_clean["Fecha"] >= "2017-01-02"]
    df_sob_cmt = build_cmt(df_clean)
    df_sob_cmt = df_sob_cmt.dropna()
    df_sob_cmt = df_sob_cmt.set_index("Fecha")
    print(f"\nSoberanos CMT (real data): {len(df_sob_cmt):,} dates")

print(f"  Columns: {list(df_sob_cmt.columns)}")
print(f"  Date range: {df_sob_cmt.index.min().date()} to {df_sob_cmt.index.max().date()}")

# Soberanos yields in bps
sob_yields_bps = df_sob_cmt * 100

# ── 1E. LC SPREADS (Soberano CMT - UST CMT) ────────────────────────────────
# Align dates between Soberanos and UST (inner join)
common_dates = df_sob_cmt.index.intersection(df_ust_cmt.index)
df_lcs = (df_sob_cmt.loc[common_dates] - df_ust_cmt.loc[common_dates]) * 100  # bps
df_lcs.columns = [f"LCS_{c}" for c in df_lcs.columns]
print(f"\nLC Spreads: {len(df_lcs):,} dates  |  Columns: {list(df_lcs.columns)}")

# ── 1F. SOBERANO INTER-NODE SPREADS ─────────────────────────────────────────
sob_spread_pairs = {
    "6s10s":  ("6Y", "10Y"),
    "6s15s":  ("6Y", "15Y"),
    "7s15s":  ("7Y", "15Y"),
    "6s12s":  ("6Y", "12Y"),
    "7s10s":  ("7Y", "10Y"),
    "8s12s":  ("8Y", "12Y"),
    "9s15s":  ("9Y", "15Y"),
    "10s15s": ("10Y", "15Y"),
}

sob_spreads = pd.DataFrame(index=df_sob_cmt.index)
for name, (short, long) in sob_spread_pairs.items():
    sob_spreads[name] = (df_sob_cmt[long] - df_sob_cmt[short]) * 100  # bps

# Belly vs 10s Sector: avg(6,7,8,9) vs avg(10,11,12)
sob_spreads["belly_vs_10s"] = (
    df_sob_cmt[["10Y", "11Y", "12Y"]].mean(axis=1)
    - df_sob_cmt[["6Y", "7Y", "8Y", "9Y"]].mean(axis=1)
) * 100

# Belly vs Long: avg(6,7,8,9) vs avg(13,14,15)
sob_spreads["belly_vs_long"] = (
    df_sob_cmt[["13Y", "14Y", "15Y"]].mean(axis=1)
    - df_sob_cmt[["6Y", "7Y", "8Y", "9Y"]].mean(axis=1)
) * 100

print(f"\nSoberano inter-node spreads: {list(sob_spreads.columns)}")

# LC Spread pairs (same pairs but on LC Spread level)
lcs_spread_pairs = {
    "LCS_6s10s":  ("LCS_6Y", "LCS_10Y"),
    "LCS_6s15s":  ("LCS_6Y", "LCS_15Y"),
    "LCS_7s15s":  ("LCS_7Y", "LCS_15Y"),
    "LCS_6s12s":  ("LCS_6Y", "LCS_12Y"),
}

lcs_spreads = pd.DataFrame(index=df_lcs.index)
for name, (short, long) in lcs_spread_pairs.items():
    lcs_spreads[name] = df_lcs[long] - df_lcs[short]  # already in bps

# LC Spread belly vs 10s sector
lcs_spreads["LCS_belly_vs_10s"] = (
    df_lcs[["LCS_10Y", "LCS_11Y", "LCS_12Y"]].mean(axis=1)
    - df_lcs[["LCS_6Y", "LCS_7Y", "LCS_8Y", "LCS_9Y"]].mean(axis=1)
)

# LC Spread belly vs long
lcs_spreads["LCS_belly_vs_long"] = (
    df_lcs[["LCS_13Y", "LCS_14Y", "LCS_15Y"]].mean(axis=1)
    - df_lcs[["LCS_6Y", "LCS_7Y", "LCS_8Y", "LCS_9Y"]].mean(axis=1)
)

print(f"LC Spread inter-node spreads: {list(lcs_spreads.columns)}")

# ── SUMMARY ──────────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("DATA LOADING COMPLETE")
print("=" * 70)
print(f"  df_ust         : UST raw yields (%)         {df_ust.shape}")
print(f"  df_ust_cmt     : UST MC-interp 6-15Y (%)    {df_ust_cmt.shape}")
print(f"  ust_yields_bps : UST yields (bps)           {ust_yields_bps.shape}")
print(f"  ust_spreads    : UST standard spreads (bps)  {ust_spreads.shape}")
print(f"  df_sob_cmt     : Soberanos CMT 6-15Y (%)    {df_sob_cmt.shape}")
print(f"  sob_yields_bps : Soberanos yields (bps)     {sob_yields_bps.shape}")
print(f"  sob_spreads    : Soberano spreads (bps)     {sob_spreads.shape}")
print(f"  df_lcs         : LC Spread levels per node (bps)  {df_lcs.shape}")
print(f"  lcs_spreads    : LC Spread pairs (bps)     {lcs_spreads.shape}")


UST raw loaded: 1990-01-02 to 2026-03-09
  Shape: (9051, 8)  |  Columns: ['1Y', '2Y', '3Y', '5Y', '7Y', '10Y', '20Y', '30Y']

UST CMT (MC-interpolated 6-15Y): 9,050 dates
  Columns: ['6Y', '7Y', '8Y', '9Y', '10Y', '11Y', '12Y', '13Y', '14Y', '15Y']

UST spreads computed: ['2s5s', '2s10s', '2s30s', '5s10s', '5s30s', '10s30s']

Soberanos CMT (dummy, MC-interpolated): 9,051 dates
  Columns: ['6Y', '7Y', '8Y', '9Y', '10Y', '11Y', '12Y', '13Y', '14Y', '15Y']
  Date range: 1990-01-02 to 2026-03-09

ASW spreads: 9,050 dates  |  Columns: ['ASW_6Y', 'ASW_7Y', 'ASW_8Y', 'ASW_9Y', 'ASW_10Y', 'ASW_11Y', 'ASW_12Y', 'ASW_13Y', 'ASW_14Y', 'ASW_15Y']

Soberano inter-node spreads: ['6s10s', '6s15s', '7s15s', '6s12s', '7s10s', '8s12s', '9s15s', '10s15s', 'belly_vs_10s', 'belly_vs_long']
ASW inter-node spreads: ['ASW_6s10s', 'ASW_6s15s', 'ASW_7s15s', 'ASW_6s12s', 'ASW_belly_vs_10s', 'ASW_belly_vs_long']

DATA LOADING COMPLETE
  df_ust         : UST raw yields (%)         (9051, 8)
  df_ust_cmt     : UST 

In [3]:
# =============================================================================
# CELL 2 – REGIME CLASSIFICATION & HELPER FUNCTIONS
# =============================================================================
# Reusable for both UST and Soberano regime detection.
# =============================================================================

REGIME_COLORS = {
    "Bull Steepener":   "#4A90D9",
    "Bear Steepener":   "#F5A623",
    "Steepener Twist":  "#7ED321",
    "Bull Flattener":   "#D0021B",
    "Bear Flattener":   "#9B59B6",
    "Flattener Twist":  "#8B6914",
}

REGIME_ORDER = [
    "Bull Steepener", "Bear Steepener", "Steepener Twist",
    "Bull Flattener", "Bear Flattener", "Flattener Twist",
]

def classify_regime(df_yields_bps, short_tenor, long_tenor, lookback):
    """Classify curve moves into 6 regimes based on spread changes.
    
    Parameters
    ----------
    df_yields_bps : DataFrame, yields in bps with DatetimeIndex
    short_tenor   : str, column name for short end (e.g. '2Y')
    long_tenor    : str, column name for long end (e.g. '10Y')
    lookback      : int, business days for change window
    
    Returns
    -------
    DataFrame with columns: spread_level, short_chg, long_chg, spread_chg, regime
    """
    result = pd.DataFrame(index=df_yields_bps.index)
    result["spread_level"] = df_yields_bps[long_tenor] - df_yields_bps[short_tenor]
    result["short_chg"]  = df_yields_bps[short_tenor].diff(lookback)
    result["long_chg"]   = df_yields_bps[long_tenor].diff(lookback)
    result["spread_chg"] = result["long_chg"] - result["short_chg"]
    result = result.dropna()

    conditions = [
        (result["spread_chg"] > 0) & (result["short_chg"] <= 0) & (result["long_chg"] <= 0),
        (result["spread_chg"] > 0) & (result["short_chg"] > 0)  & (result["long_chg"] > 0),
        (result["spread_chg"] > 0) & (result["short_chg"] <= 0) & (result["long_chg"] > 0),
        (result["spread_chg"] < 0) & (result["short_chg"] <= 0) & (result["long_chg"] <= 0),
        (result["spread_chg"] < 0) & (result["short_chg"] > 0)  & (result["long_chg"] > 0),
        (result["spread_chg"] < 0) & (result["short_chg"] > 0)  & (result["long_chg"] <= 0),
    ]
    labels = REGIME_ORDER
    result["regime"] = np.select(conditions, labels, default="")
    result["regime"] = result["regime"].replace("", np.nan).ffill().bfill()
    return result


def classify_regime_composite(df_yields_bps, short_cols, long_cols, lookback):
    """Classify regime using average of multiple tenor columns (belly vs sector).
    
    Parameters
    ----------
    df_yields_bps : DataFrame, yields in bps
    short_cols    : list of str, columns for short/belly leg
    long_cols     : list of str, columns for long leg
    lookback      : int, business days
    
    Returns
    -------
    DataFrame with same structure as classify_regime output.
    """
    short_avg = df_yields_bps[short_cols].mean(axis=1)
    long_avg  = df_yields_bps[long_cols].mean(axis=1)
    
    composite = pd.DataFrame(index=df_yields_bps.index)
    composite["short_avg"] = short_avg
    composite["long_avg"]  = long_avg
    
    result = pd.DataFrame(index=df_yields_bps.index)
    result["spread_level"] = long_avg - short_avg
    result["short_chg"]  = short_avg.diff(lookback)
    result["long_chg"]   = long_avg.diff(lookback)
    result["spread_chg"] = result["long_chg"] - result["short_chg"]
    result = result.dropna()

    conditions = [
        (result["spread_chg"] > 0) & (result["short_chg"] <= 0) & (result["long_chg"] <= 0),
        (result["spread_chg"] > 0) & (result["short_chg"] > 0)  & (result["long_chg"] > 0),
        (result["spread_chg"] > 0) & (result["short_chg"] <= 0) & (result["long_chg"] > 0),
        (result["spread_chg"] < 0) & (result["short_chg"] <= 0) & (result["long_chg"] <= 0),
        (result["spread_chg"] < 0) & (result["short_chg"] > 0)  & (result["long_chg"] > 0),
        (result["spread_chg"] < 0) & (result["short_chg"] > 0)  & (result["long_chg"] <= 0),
    ]
    result["regime"] = np.select(conditions, REGIME_ORDER, default="")
    result["regime"] = result["regime"].replace("", np.nan).ffill().bfill()
    return result


def compute_streak_days(regimes_series):
    """Count consecutive days each regime has been active."""
    streaks = []
    count = 1
    prev = regimes_series.iloc[0]
    streaks.append(count)
    for i in range(1, len(regimes_series)):
        if regimes_series.iloc[i] == prev:
            count += 1
        else:
            count = 1
            prev = regimes_series.iloc[i]
        streaks.append(count)
    return streaks


def compute_avg_durations(regimes_series):
    """Average streak length per regime."""
    streaks = []
    current = regimes_series.iloc[0]
    length = 1
    for i in range(1, len(regimes_series)):
        if regimes_series.iloc[i] == current:
            length += 1
        else:
            streaks.append((current, length))
            current = regimes_series.iloc[i]
            length = 1
    streaks.append((current, length))
    streak_df = pd.DataFrame(streaks, columns=["regime", "duration"])
    return streak_df.groupby("regime")["duration"].mean()


def regime_summary(regimes_df, weeks=None):
    """Compute regime distribution statistics."""
    data = regimes_df.copy()
    if weeks:
        data = data.iloc[-(weeks * 5):]
    summary = data.groupby("regime").agg(
        count=("spread_chg", "size"),
        avg_spread_chg=("spread_chg", "mean"),
        avg_short_chg=("short_chg", "mean"),
        avg_long_chg=("long_chg", "mean"),
    ).round(2)
    summary["pct"] = (summary["count"] / summary["count"].sum() * 100).round(1)
    return summary.sort_values("count", ascending=False)


def get_regime_streaks_df(regimes_series):
    """Return DataFrame of all regime streaks with start/end dates."""
    streaks = []
    current = regimes_series.iloc[0]
    start_idx = 0
    for i in range(1, len(regimes_series)):
        if regimes_series.iloc[i] != current:
            streaks.append({
                "regime": current,
                "start": regimes_series.index[start_idx],
                "end": regimes_series.index[i - 1],
                "duration": i - start_idx,
            })
            current = regimes_series.iloc[i]
            start_idx = i
    streaks.append({
        "regime": current,
        "start": regimes_series.index[start_idx],
        "end": regimes_series.index[-1],
        "duration": len(regimes_series) - start_idx,
    })
    return pd.DataFrame(streaks)


print("Regime functions loaded OK")

# Quick test with UST 2s10s
ust_regimes_2s10s = classify_regime(ust_yields_bps, "2Y", "10Y", lookback=5)
print(f"\nUST 2s10s regime test (5d lookback): {len(ust_regimes_2s10s):,} obs")
print(ust_regimes_2s10s["regime"].value_counts())


Regime functions loaded OK

UST 2s10s regime test (5d lookback): 9,044 obs
regime
Bull Flattener     2381
Bear Steepener     1894
Bull Steepener     1576
Bear Flattener     1399
Flattener Twist     935
Steepener Twist     859
Name: count, dtype: int64


In [4]:
# =============================================================================
# CELL 3 – UST REGIME CHART (Interactive, Discrete Dates, Multi-Spread)
# =============================================================================
import ipywidgets as widgets
from IPython.display import display, clear_output
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# UST spread options (for regime detection)
UST_SPREAD_OPTIONS = {
    "2s5s":   ("2Y", "5Y"),
    "2s10s":  ("2Y", "10Y"),
    "2s30s":  ("2Y", "30Y"),
    "5s10s":  ("5Y", "10Y"),
    "5s30s":  ("5Y", "30Y"),
    "10s30s": ("10Y", "30Y"),
}

def build_ust_regime_chart(spread_name, lookback, weeks, avg_window_years):
    """Build interactive Plotly regime chart for a UST spread."""
    short_t, long_t = UST_SPREAD_OPTIONS[spread_name]
    reg = classify_regime(ust_yields_bps, short_t, long_t, lookback)

    # Avg duration window
    if avg_window_years == "Max":
        reg_for_avg = reg
    else:
        n_days_avg = int(float(avg_window_years) * 252)
        reg_for_avg = reg.iloc[-n_days_avg:]
    avg_dur = compute_avg_durations(reg_for_avg["regime"])

    # Streaks on full series
    reg["streak_days"] = compute_streak_days(reg["regime"])

    # Trim to display window
    n_days = weeks * 5
    plot_data = reg.iloc[-n_days:].copy().reset_index()

    plot_data["avg_duration"] = plot_data["regime"].map(avg_dur).round(1)
    plot_data["dist_to_avg"] = (plot_data["streak_days"] - plot_data["avg_duration"]).round(1)
    bar_colors = plot_data["regime"].map(REGIME_COLORS).tolist()

    # Use integer x-axis for discrete dates (no weekend gaps)
    x_idx = list(range(len(plot_data)))

    # Hover text
    hover_text = []
    for _, row in plot_data.iterrows():
        hover_text.append(
            f"Date: {row['Fecha'].strftime('%d %b %Y')}<br>"
            f"Spread: {row['spread_level']:.1f} bps<br>"
            f"Regime: {row['regime']}<br>"
            f"Days in regime: {int(row['streak_days'])}<br>"
            f"Avg duration: {row['avg_duration']:.1f}d<br>"
            f"Dist to avg: {row['dist_to_avg']:+.1f}d"
        )

    fig = go.Figure()

    # Bars (colored by regime)
    fig.add_trace(go.Bar(
        x=x_idx, y=plot_data["spread_level"],
        marker_color=bar_colors,
        hovertext=hover_text, hoverinfo="text",
        showlegend=False,
    ))

    # Line overlay
    fig.add_trace(go.Scatter(
        x=x_idx, y=plot_data["spread_level"],
        mode="lines", line=dict(color="#F5A623", width=1.5),
        hoverinfo="skip", name=f"{spread_name} Curve",
    ))

    # Legend entries
    for regime_name in REGIME_ORDER:
        fig.add_trace(go.Bar(
            x=[None], y=[None],
            marker_color=REGIME_COLORS[regime_name],
            name=regime_name, showlegend=True,
        ))

    # X-axis tick labels (discrete dates, ~15 labels)
    n_ticks = min(15, len(plot_data))
    tick_idx = np.linspace(0, len(plot_data) - 1, n_ticks, dtype=int)

    # Current regime info
    current_regime = plot_data["regime"].iloc[-1]
    current_streak = int(plot_data["streak_days"].iloc[-1])
    last_date = plot_data["Fecha"].iloc[-1].strftime("%Y-%m-%d")

    window_label = "Max" if avg_window_years == "Max" else f"{avg_window_years}y"

    fig.update_layout(
        title=dict(
            text=(
                f"<b>{spread_name}</b>   "
                f"Lookback: {lookback}d  |  Display: {weeks}w  |  Avg: {window_label}<br>"
                f"<span style='font-size:11px'>Current ({last_date}): "
                f"<b>{current_regime}</b> for {current_streak}d</span>"
            ),
            font=dict(color="black", size=13),
        ),
        paper_bgcolor="#ffffff", plot_bgcolor="#ffffff",
        font=dict(color="black"),
        yaxis=dict(title="bps", gridcolor="#ddd", gridwidth=0.5),
        xaxis=dict(
            tickvals=tick_idx.tolist(),
            ticktext=[plot_data["Fecha"].iloc[i].strftime("%b %y") for i in tick_idx],
            gridcolor="#ddd", gridwidth=0.5,
        ),
        legend=dict(
            bgcolor="rgba(255,255,255,0.9)", bordercolor="#ccc",
            borderwidth=1, font=dict(size=9),
        ),
        dragmode="zoom", height=420, bargap=0,
        margin=dict(t=70, b=40, l=50, r=20),
    )
    fig.show()

    # Print regime stats
    stats = regime_summary(reg.iloc[-n_days:])
    print(f"\nRegime stats (last {weeks}w):")
    print(stats.to_string())


def build_ust_multi_panel(lookback, weeks, avg_window_years, spread_list=None):
    """Build a multi-spread panel showing all UST spreads simultaneously."""
    if spread_list is None:
        spread_list = list(UST_SPREAD_OPTIONS.keys())
    
    n = len(spread_list)
    fig = make_subplots(
        rows=n, cols=1, shared_xaxes=True,
        subplot_titles=[f"<b>{s}</b>" for s in spread_list],
        vertical_spacing=0.03,
    )
    
    # Use the shortest series for x-axis alignment
    n_days = weeks * 5
    
    for row_idx, spread_name in enumerate(spread_list, 1):
        short_t, long_t = UST_SPREAD_OPTIONS[spread_name]
        reg = classify_regime(ust_yields_bps, short_t, long_t, lookback)
        reg["streak_days"] = compute_streak_days(reg["regime"])
        plot_data = reg.iloc[-n_days:].copy().reset_index()
        
        x_idx = list(range(len(plot_data)))
        bar_colors = plot_data["regime"].map(REGIME_COLORS).tolist()
        
        hover_text = [
            f"{row['Fecha'].strftime('%d %b %Y')}<br>"
            f"{spread_name}: {row['spread_level']:.1f} bps<br>"
            f"Regime: {row['regime']} ({int(row['streak_days'])}d)"
            for _, row in plot_data.iterrows()
        ]
        
        fig.add_trace(go.Bar(
            x=x_idx, y=plot_data["spread_level"],
            marker_color=bar_colors,
            hovertext=hover_text, hoverinfo="text",
            showlegend=False,
        ), row=row_idx, col=1)
        
        fig.add_trace(go.Scatter(
            x=x_idx, y=plot_data["spread_level"],
            mode="lines", line=dict(color="#F5A623", width=1),
            hoverinfo="skip", showlegend=False,
        ), row=row_idx, col=1)
        
        fig.update_yaxes(title_text="bps", gridcolor="#ddd", row=row_idx, col=1)
    
    # X-axis ticks on bottom subplot only
    n_ticks = min(12, n_days)
    tick_idx = np.linspace(0, n_days - 1, n_ticks, dtype=int)
    # Use dates from last spread's plot_data
    fig.update_xaxes(
        tickvals=tick_idx.tolist(),
        ticktext=[plot_data["Fecha"].iloc[min(i, len(plot_data)-1)].strftime("%b %y") for i in tick_idx],
        row=n, col=1,
    )
    
    # Legend (add once)
    for regime_name in REGIME_ORDER:
        fig.add_trace(go.Bar(
            x=[None], y=[None],
            marker_color=REGIME_COLORS[regime_name],
            name=regime_name, showlegend=True,
        ), row=1, col=1)

    fig.update_layout(
        title=dict(
            text=f"<b>UST Curve Regime Panel</b>  |  Lookback: {lookback}d  |  Display: {weeks}w",
            font=dict(color="black", size=14),
        ),
        paper_bgcolor="#ffffff", plot_bgcolor="#ffffff",
        font=dict(color="black", size=10),
        legend=dict(
            bgcolor="rgba(255,255,255,0.9)", bordercolor="#ccc",
            borderwidth=1, font=dict(size=9),
            orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1,
        ),
        height=220 * n, bargap=0,
        margin=dict(t=80, b=40, l=50, r=20),
    )
    for ann in fig.layout.annotations:
        ann.font.color = "black"
        ann.font.size = 11
    
    fig.show()


# ── UST REGIME WIDGET ───────────────────────────────────────────────────────
w_ust_spread = widgets.Dropdown(
    options=["PANEL"] + list(UST_SPREAD_OPTIONS.keys()),
    value="2s10s",
    description="UST Spread:",
    style={"description_width": "90px"},
)
w_ust_lookback = widgets.Text(value="20", description="Lookback (d):",
    style={"description_width": "100px"}, layout=widgets.Layout(width="180px"))
w_ust_weeks = widgets.Text(value="52", description="Display (w):",
    style={"description_width": "100px"}, layout=widgets.Layout(width="180px"))
w_ust_avg = widgets.Text(value="3", description="Avg Window (y):",
    style={"description_width": "110px"}, layout=widgets.Layout(width="200px"))
w_ust_btn = widgets.Button(description="Update", button_style="info",
    layout=widgets.Layout(width="90px"))
ust_output = widgets.Output()

def on_ust_click(_):
    try: lb = max(5, min(180, int(w_ust_lookback.value)))
    except: lb = 20
    try: wk = max(1, min(260, int(w_ust_weeks.value)))
    except: wk = 52
    avg_val = w_ust_avg.value.strip().lower()
    avg_w = "Max" if avg_val == "max" else str(max(0.25, float(avg_val))) if avg_val.replace(".", "").isdigit() else "3"

    with ust_output:
        clear_output(wait=True)
        if w_ust_spread.value == "PANEL":
            build_ust_multi_panel(lb, wk, avg_w)
        else:
            build_ust_regime_chart(w_ust_spread.value, lb, wk, avg_w)

w_ust_btn.on_click(on_ust_click)

display(widgets.HBox([w_ust_spread, w_ust_lookback, w_ust_weeks, w_ust_avg, w_ust_btn]))
display(ust_output)
on_ust_click(None)


Output()

In [5]:
# =============================================================================
# CELL 4 – SOBERANO REGIME DETECTION (Absolute Yield & LC Spread)
# =============================================================================
# Regime detection for Soberanos on:
#   - Absolute yield spreads: 6s10s, 6s15s, 7s15s, 6s12s, belly_vs_10s, belly_vs_long
#   - LC Spreads: same pairs but computed on LC Spread (Sob CMT - UST CMT)
# =============================================================================

# Soberano spread options for regime detection
SOB_REGIME_SPREADS = {
    # Simple 2-tenor spreads (absolute yield)
    "6s10s (Yield)":  {"type": "simple", "data": "yield", "short": "6Y",  "long": "10Y"},
    "6s15s (Yield)":  {"type": "simple", "data": "yield", "short": "6Y",  "long": "15Y"},
    "7s15s (Yield)":  {"type": "simple", "data": "yield", "short": "7Y",  "long": "15Y"},
    "6s12s (Yield)":  {"type": "simple", "data": "yield", "short": "6Y",  "long": "12Y"},
    # Composite spreads (absolute yield)
    "Belly vs 10s Sector (Yield)": {
        "type": "composite", "data": "yield",
        "short_cols": ["6Y", "7Y", "8Y", "9Y"],
        "long_cols":  ["10Y", "11Y", "12Y"],
    },
    "Belly vs Long (Yield)": {
        "type": "composite", "data": "yield",
        "short_cols": ["6Y", "7Y", "8Y", "9Y"],
        "long_cols":  ["13Y", "14Y", "15Y"],
    },
    # Simple 2-tenor spreads (LC Spread)
    "6s10s (LC Spread)":  {"type": "simple", "data": "lcs", "short": "LCS_6Y",  "long": "LCS_10Y"},
    "6s15s (LC Spread)":  {"type": "simple", "data": "lcs", "short": "LCS_6Y",  "long": "LCS_15Y"},
    "7s15s (LC Spread)":  {"type": "simple", "data": "lcs", "short": "LCS_7Y",  "long": "LCS_15Y"},
    "6s12s (LC Spread)":  {"type": "simple", "data": "lcs", "short": "LCS_6Y",  "long": "LCS_12Y"},
    # Composite spreads (LC Spread)
    "Belly vs 10s Sector (LC Spread)": {
        "type": "composite", "data": "lcs",
        "short_cols": ["LCS_6Y", "LCS_7Y", "LCS_8Y", "LCS_9Y"],
        "long_cols":  ["LCS_10Y", "LCS_11Y", "LCS_12Y"],
    },
    "Belly vs Long (LC Spread)": {
        "type": "composite", "data": "lcs",
        "short_cols": ["LCS_6Y", "LCS_7Y", "LCS_8Y", "LCS_9Y"],
        "long_cols":  ["LCS_13Y", "LCS_14Y", "LCS_15Y"],
    },
}


def get_sob_regime(spread_key, lookback):
    """Compute regime for a given Soberano spread configuration."""
    cfg = SOB_REGIME_SPREADS[spread_key]
    
    # Select the right data source
    if cfg["data"] == "yield":
        data_bps = sob_yields_bps
    else:  # lcs
        data_bps = df_lcs  # already in bps
    
    if cfg["type"] == "simple":
        return classify_regime(data_bps, cfg["short"], cfg["long"], lookback)
    else:  # composite
        return classify_regime_composite(
            data_bps, cfg["short_cols"], cfg["long_cols"], lookback
        )


def build_sob_regime_chart(spread_key, lookback, weeks, avg_window_years):
    """Build Plotly regime chart for a Soberano spread."""
    reg = get_sob_regime(spread_key, lookback)
    
    if avg_window_years == "Max":
        reg_for_avg = reg
    else:
        n_days_avg = int(float(avg_window_years) * 252)
        reg_for_avg = reg.iloc[-n_days_avg:]
    avg_dur = compute_avg_durations(reg_for_avg["regime"])

    reg["streak_days"] = compute_streak_days(reg["regime"])
    n_days = weeks * 5
    plot_data = reg.iloc[-n_days:].copy().reset_index()

    plot_data["avg_duration"] = plot_data["regime"].map(avg_dur).round(1)
    plot_data["dist_to_avg"] = (plot_data["streak_days"] - plot_data["avg_duration"]).round(1)
    bar_colors = plot_data["regime"].map(REGIME_COLORS).tolist()
    x_idx = list(range(len(plot_data)))

    hover_text = [
        f"Date: {row['Fecha'].strftime('%d %b %Y')}<br>"
        f"Spread: {row['spread_level']:.1f} bps<br>"
        f"Regime: {row['regime']}<br>"
        f"Days: {int(row['streak_days'])}  |  Avg: {row['avg_duration']:.1f}d<br>"
        f"Dist to avg: {row['dist_to_avg']:+.1f}d"
        for _, row in plot_data.iterrows()
    ]

    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=x_idx, y=plot_data["spread_level"],
        marker_color=bar_colors, hovertext=hover_text, hoverinfo="text",
        showlegend=False,
    ))
    fig.add_trace(go.Scatter(
        x=x_idx, y=plot_data["spread_level"],
        mode="lines", line=dict(color="#F5A623", width=1.5),
        hoverinfo="skip", name=f"{spread_key}",
    ))
    for regime_name in REGIME_ORDER:
        fig.add_trace(go.Bar(
            x=[None], y=[None],
            marker_color=REGIME_COLORS[regime_name],
            name=regime_name, showlegend=True,
        ))

    n_ticks = min(15, len(plot_data))
    tick_idx = np.linspace(0, len(plot_data) - 1, n_ticks, dtype=int)

    current_regime = plot_data["regime"].iloc[-1]
    current_streak = int(plot_data["streak_days"].iloc[-1])
    last_date = plot_data["Fecha"].iloc[-1].strftime("%Y-%m-%d")
    window_label = "Max" if avg_window_years == "Max" else f"{avg_window_years}y"

    fig.update_layout(
        title=dict(
            text=(
                f"<b>Soberanos: {spread_key}</b>   "
                f"Lookback: {lookback}d  |  Display: {weeks}w  |  Avg: {window_label}<br>"
                f"<span style='font-size:11px'>Current ({last_date}): "
                f"<b>{current_regime}</b> for {current_streak}d</span>"
            ),
            font=dict(color="black", size=13),
        ),
        paper_bgcolor="#ffffff", plot_bgcolor="#ffffff",
        font=dict(color="black"),
        yaxis=dict(title="bps", gridcolor="#ddd", gridwidth=0.5),
        xaxis=dict(
            tickvals=tick_idx.tolist(),
            ticktext=[plot_data["Fecha"].iloc[i].strftime("%b %y") for i in tick_idx],
            gridcolor="#ddd",
        ),
        legend=dict(
            bgcolor="rgba(255,255,255,0.9)", bordercolor="#ccc",
            borderwidth=1, font=dict(size=9),
        ),
        dragmode="zoom", height=420, bargap=0,
        margin=dict(t=70, b=40, l=50, r=20),
    )
    fig.show()

    stats = regime_summary(reg.iloc[-n_days:])
    print(f"\nSoberano regime stats ({spread_key}, last {weeks}w):")
    print(stats.to_string())


# ── SOBERANO REGIME WIDGET ──────────────────────────────────────────────────
w_sob_spread = widgets.Dropdown(
    options=list(SOB_REGIME_SPREADS.keys()),
    value="6s10s (Yield)",
    description="Sob Spread:",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="280px"),
)
w_sob_lookback = widgets.Text(value="20", description="Lookback (d):",
    style={"description_width": "100px"}, layout=widgets.Layout(width="180px"))
w_sob_weeks = widgets.Text(value="52", description="Display (w):",
    style={"description_width": "100px"}, layout=widgets.Layout(width="180px"))
w_sob_avg = widgets.Text(value="3", description="Avg Window (y):",
    style={"description_width": "110px"}, layout=widgets.Layout(width="200px"))
w_sob_btn = widgets.Button(description="Update", button_style="info",
    layout=widgets.Layout(width="90px"))
sob_regime_output = widgets.Output()

def on_sob_regime_click(_):
    try: lb = max(5, min(180, int(w_sob_lookback.value)))
    except: lb = 20
    try: wk = max(1, min(260, int(w_sob_weeks.value)))
    except: wk = 52
    avg_val = w_sob_avg.value.strip().lower()
    avg_w = "Max" if avg_val == "max" else str(max(0.25, float(avg_val))) if avg_val.replace(".", "").isdigit() else "3"

    with sob_regime_output:
        clear_output(wait=True)
        build_sob_regime_chart(w_sob_spread.value, lb, wk, avg_w)

w_sob_btn.on_click(on_sob_regime_click)

display(widgets.HBox([w_sob_spread, w_sob_lookback, w_sob_weeks, w_sob_avg, w_sob_btn]))
display(sob_regime_output)
on_sob_regime_click(None)


Output()

In [6]:
# =============================================================================
# CELL 5 – SOBERANO EVOLUTION MONITOR
# =============================================================================
# Interactive widget showing evolution of Soberano yields, spreads, butterflies
# at both CMT and LC Spread levels, with configurable lookback
# change windows and date range selection.
# =============================================================================
import ipywidgets as widgets
from IPython.display import display, clear_output
import plotly.graph_objects as go

# Available Soberano tenors for input boxes
SOB_TENOR_OPTIONS = [str(t) for t in SOB_CMT_TENORS]  # ["6","7",..."15"]

# ── HELPER: COMPUTE SOBERANO METRIC SERIES ──────────────────────────────────

def compute_sob_metric(product, metric_type, tenors):
    """Compute a Soberano metric time series (in bps).

    Parameters
    ----------
    product     : str, "CMT Tenor" or "LC Spread"
    metric_type : str, "Level", "Spread", or "Butterfly"
    tenors      : list of int, tenor inputs (1, 2, or 3 values)

    Returns
    -------
    (pd.Series with DatetimeIndex, str label)
    """
    if product == "CMT Tenor":
        src = sob_yields_bps          # yields in bps
        tag = "Yield"
        col = lambda t: f"{t}Y"
    else:                             # LC Spread
        src = df_lcs                  # LC Spread already in bps
        tag = "LC Spread"
        col = lambda t: f"LCS_{t}Y"

    if metric_type == "Level":
        series = src[col(tenors[0])]
        label = f"{tenors[0]}Y {tag}"
    elif metric_type == "Spread":
        series = src[col(tenors[1])] - src[col(tenors[0])]
        label = f"{tenors[0]}s{tenors[1]}s {tag}"
    else:  # Butterfly = wing1 + wing2 - 2*belly
        series = src[col(tenors[0])] + src[col(tenors[2])] - 2 * src[col(tenors[1])]
        label = f"{tenors[0]}s{tenors[1]}s{tenors[2]}s BF {tag}"

    return series.dropna(), label


# ── CHART BUILDER ────────────────────────────────────────────────────────────

def build_evolution_chart(product, metric_type, tenors, lookback_days,
                          start_date, end_date, ytick_step=None,
                          show_hla=False, show_ma=False, ma_window=20):
    """Build Plotly chart: level line + rolling change bars (left axis).
    White background, no green/red bars, zoom-enabled."""
    series, label = compute_sob_metric(product, metric_type, tenors)

    # Filter to date range
    mask = (series.index >= pd.Timestamp(start_date)) & (
            series.index <= pd.Timestamp(end_date))
    series = series[mask]
    if len(series) < lookback_days + 1:
        print("Not enough data for selected parameters.")
        return

    # Rolling change
    chg = series.diff(lookback_days).dropna()
    level = series.loc[chg.index]

    # Discrete x-axis (no weekend / holiday gaps)
    x_idx = list(range(len(chg)))
    dates = chg.index

    fig = go.Figure()

    # Level line (right y-axis)
    fig.add_trace(go.Scatter(
        x=x_idx, y=level.values,
        mode="lines", line=dict(color="#4A90D9", width=1.5),
        name=f"{label} Level", yaxis="y2",
        hovertemplate="%{customdata}<br>Level: %{y:.1f} bps<extra></extra>",
        customdata=[d.strftime("%d %b %Y") for d in dates],
    ))

    # Change line (left y-axis) — replaces green/red bars
    fig.add_trace(go.Scatter(
        x=x_idx, y=chg.values,
        mode="lines", line=dict(color="#9B59B6", width=1.2),
        name=f"{lookback_days}d Change",
        hovertemplate="%{customdata}<br>\u0394: %{y:+.1f} bps<extra></extra>",
        customdata=[d.strftime("%d %b %Y") for d in dates],
    ))

    # Y-axis label depends on product + metric_type
    if product == "CMT Tenor" and metric_type == "Level":
        right_y_title = "Yield (%)"
        right_y_vals = level.values / 100.0
        fig.data[0].y = right_y_vals.tolist()
        fig.data[0].hovertemplate = "%{customdata}<br>Yield: %{y:.2f}%<extra></extra>"
    else:
        right_y_title = "Level (bps)"

    # Optional: Hi/Lo/Avg horizontal lines (full date range, not affected by zoom)
    if show_hla:
        hi_val = level.max()
        lo_val = level.min()
        avg_val = level.mean()
        if product == "CMT Tenor" and metric_type == "Level":
            hi_val /= 100.0
            lo_val /= 100.0
            avg_val /= 100.0
        fig.add_hline(y=float(hi_val), line_dash="dot", line_color="#D0021B",
                      line_width=1, annotation_text=f"High: {hi_val:.2f}",
                      annotation_position="top right",
                      annotation_font_color="#D0021B",
                      annotation_font_size=9, yref="y2")
        fig.add_hline(y=float(lo_val), line_dash="dot", line_color="#4A90D9",
                      line_width=1, annotation_text=f"Low: {lo_val:.2f}",
                      annotation_position="bottom right",
                      annotation_font_color="#4A90D9",
                      annotation_font_size=9, yref="y2")
        fig.add_hline(y=float(avg_val), line_dash="dash", line_color="#F5A623",
                      line_width=1, annotation_text=f"Avg: {avg_val:.2f}",
                      annotation_position="top right",
                      annotation_font_color="#F5A623",
                      annotation_font_size=9, yref="y2")

    # Optional: Moving Average line
    if show_ma and ma_window > 0:
        if product == "CMT Tenor" and metric_type == "Level":
            ma_series = pd.Series(right_y_vals, index=dates).rolling(ma_window, min_periods=1).mean()
        else:
            ma_series = level.rolling(ma_window, min_periods=1).mean()
        fig.add_trace(go.Scatter(
            x=x_idx, y=ma_series.values,
            mode="lines", line=dict(color="#F5A623", width=1.5, dash="dash"),
            name=f"{ma_window}d MA", yaxis="y2",
            hovertemplate="%{customdata}<br>MA: %{y:.2f}<extra></extra>",
            customdata=[d.strftime("%d %b %Y") for d in dates],
        ))

    # Tick labels (~15 evenly spaced)
    n_ticks = min(15, len(x_idx))
    tick_pos = np.linspace(0, len(x_idx) - 1, n_ticks, dtype=int)

    lb_label = "1d" if lookback_days == 1 else f"{lookback_days}d"
    last_val = level.iloc[-1]
    last_chg = chg.iloc[-1]
    last_dt = dates[-1].strftime("%Y-%m-%d")

    if product == "CMT Tenor" and metric_type == "Level":
        val_str = f"Yield {last_val/100:.2f}%"
    else:
        val_str = f"Level {last_val:.1f} bps"

    y2_config = dict(title=right_y_title, overlaying="y", side="right",
                     showgrid=False)
    if ytick_step and ytick_step > 0:
        y2_config["dtick"] = ytick_step if not (product == "CMT Tenor" and metric_type == "Level") else ytick_step / 100.0

    fig.update_layout(
        title=dict(
            text=(
                f"<b>Soberano Evolution: {label}</b><br>"
                f"<span style='font-size:11px;color:#555'>As of {last_dt}:  "
                f"{val_str}  |  "
                f"{lb_label} chg: {last_chg:+.1f} bps</span>"
            ),
            font=dict(color="black", size=13),
            x=0.01, xanchor="left",
        ),
        paper_bgcolor="#ffffff", plot_bgcolor="#ffffff",
        font=dict(color="black"),
        yaxis=dict(title=f"{lb_label} Change (bps)", gridcolor="#ddd",
                   side="left", zeroline=True,
                   zerolinecolor="rgba(0,0,0,0.15)"),
        yaxis2=y2_config,
        xaxis=dict(
            tickvals=tick_pos.tolist(),
            ticktext=[dates[i].strftime("%b %y") for i in tick_pos],
            gridcolor="#ddd",
            rangeslider=dict(visible=True, thickness=0.06),
        ),
        legend=dict(
            bgcolor="rgba(255,255,255,0.9)", bordercolor="#ccc",
            borderwidth=1, font=dict(size=9),
            orientation="h", yanchor="top", y=-0.18,
            xanchor="center", x=0.5,
        ),
        dragmode="zoom", height=500, bargap=0.1,
        margin=dict(t=80, b=100, l=60, r=60),
    )
    fig.show()


# ── WIDGET LAYOUT ────────────────────────────────────────────────────────────

w_evo_product = widgets.Dropdown(
    options=["CMT Tenor", "LC Spread"], value="CMT Tenor",
    description="Product:", style={"description_width": "70px"},
    layout=widgets.Layout(width="220px"),
)
w_evo_type = widgets.Dropdown(
    options=["Level", "Spread", "Butterfly"], value="Spread",
    description="Type:", style={"description_width": "50px"},
    layout=widgets.Layout(width="170px"),
)
w_evo_t1 = widgets.Dropdown(
    options=SOB_TENOR_OPTIONS, value="6", description="T1:",
    style={"description_width": "30px"}, layout=widgets.Layout(width="100px"),
)
w_evo_t2 = widgets.Dropdown(
    options=SOB_TENOR_OPTIONS, value="10", description="T2:",
    style={"description_width": "30px"}, layout=widgets.Layout(width="100px"),
)
w_evo_t3 = widgets.Dropdown(
    options=SOB_TENOR_OPTIONS, value="15", description="T3:",
    style={"description_width": "30px"}, layout=widgets.Layout(width="100px"),
)

# Dynamic visibility based on Type
w_evo_t2.layout.display = "flex"   # visible for Spread
w_evo_t3.layout.display = "none"   # hidden unless Butterfly

def _evo_type_changed(change):
    w_evo_t2.layout.display = "flex" if change["new"] in ("Spread", "Butterfly") else "none"
    w_evo_t3.layout.display = "flex" if change["new"] == "Butterfly" else "none"
w_evo_type.observe(_evo_type_changed, names="value")

w_evo_lookback = widgets.Dropdown(
    options=[("Intraday (1d)", 1), ("5d", 5), ("20d", 20)], value=5,
    description="Lookback:", style={"description_width": "70px"},
    layout=widgets.Layout(width="190px"),
)

_evo_min = df_sob_cmt.index.min().date()
_evo_max = df_sob_cmt.index.max().date()
_evo_default_start = (df_sob_cmt.index.max() - pd.DateOffset(years=3)).date()

w_evo_start = widgets.DatePicker(
    value=_evo_default_start, description="Start:",
    style={"description_width": "50px"}, layout=widgets.Layout(width="200px"),
)
w_evo_end = widgets.DatePicker(
    value=_evo_max, description="End:",
    style={"description_width": "50px"}, layout=widgets.Layout(width="200px"),
)
w_evo_btn = widgets.Button(
    description="Update", button_style="info",
    layout=widgets.Layout(width="90px"),
)

# Y-axis tick step
w_evo_ytick = widgets.Text(
    value="", description="Y Tick (bps):",
    style={"description_width": "80px"}, layout=widgets.Layout(width="160px"),
    placeholder="auto",
)

# Below-chart checkboxes
w_evo_hla = widgets.Checkbox(
    value=False, description="Hi / Lo / Avg",
    layout=widgets.Layout(width="150px"),
)
w_evo_ma = widgets.Checkbox(
    value=False, description="Moving Avg",
    layout=widgets.Layout(width="140px"),
)
w_evo_ma_days = widgets.Text(
    value="20", description="MA days:",
    style={"description_width": "60px"}, layout=widgets.Layout(width="130px"),
)

evo_output = widgets.Output()

def _on_evo_click(_):
    mt = w_evo_type.value
    tenors = [int(w_evo_t1.value)]
    if mt in ("Spread", "Butterfly"):
        tenors.append(int(w_evo_t2.value))
    if mt == "Butterfly":
        tenors.append(int(w_evo_t3.value))
    try:
        yt = float(w_evo_ytick.value) if w_evo_ytick.value.strip() else None
    except:
        yt = None
    try:
        ma_d = max(1, int(w_evo_ma_days.value))
    except:
        ma_d = 20
    with evo_output:
        clear_output(wait=True)
        build_evolution_chart(
            w_evo_product.value, mt, tenors,
            w_evo_lookback.value,
            w_evo_start.value, w_evo_end.value,
            ytick_step=yt,
            show_hla=w_evo_hla.value,
            show_ma=w_evo_ma.value,
            ma_window=ma_d,
        )

w_evo_btn.on_click(_on_evo_click)

display(widgets.VBox([
    widgets.HBox([w_evo_product, w_evo_type, w_evo_t1, w_evo_t2, w_evo_t3,
                  w_evo_lookback]),
    widgets.HBox([w_evo_start, w_evo_end, w_evo_ytick, w_evo_btn]),
]))
display(evo_output)
display(widgets.HBox([w_evo_hla, w_evo_ma, w_evo_ma_days]))
_on_evo_click(None)

Output()

In [ ]:
# =============================================================================
# CELL 6 – SOBERANO CURVE-REGIME EVENT STUDY
# =============================================================================
# Plots cumulative change paths of a selected Soberano metric conditional on
# the UST curve regime.  Mirrors the "US Curve-Regime Event Study on Global
# Futures" reference chart but applied to Soberanos instead of futures.
#
# Controls:
#   Product   – CMT Tenor | LC Spread
#   Type      – Level | Spread | Butterfly (with dynamic tenor input boxes)
#   US Curve  – which UST spread drives regime detection
#   Lookback  – 5d | 20d (regime computation window)
#   Regime    – which of the 6 regimes to study
#   Band      – None | 1SD | 2SD | 3SD | 10th/90th | 25th/75th
#   Dates     – historical window for streaks
#   Show mean – toggle average path
# =============================================================================
import ipywidgets as widgets
from IPython.display import display, clear_output
import plotly.graph_objects as go

EVENT_STUDY_HORIZON = 20  # X-axis always 20 business days

# ── CORE: EXTRACT REGIME-CONDITIONAL PATHS ──────────────────────────────────

def extract_regime_paths(ust_spread, ust_lookback, regime_label,
                         sob_product, sob_type, sob_tenors,
                         start_date, end_date):
    """Identify every occurrence of *regime_label* on the UST curve and
    compute the cumulative change of the selected Soberano metric from
    the start of each regime episode.

    Returns
    -------
    dict with:
        paths          – list of np.arrays (cum Δ bps, max 21 values incl. day 0)
        path_dates     – list of lists of Timestamps
        current_path   – np.array or None (red line)
        current_info   – dict {regime, start, duration, last_date, matches}
        streaks        – DataFrame of matching streaks
        sob_label      – str, descriptive label
    """
    # 1. UST regime series
    short_t, long_t = UST_SPREAD_OPTIONS[ust_spread]
    ust_reg = classify_regime(ust_yields_bps, short_t, long_t, ust_lookback)

    # 2. All regime streaks
    all_streaks = get_regime_streaks_df(ust_reg["regime"])

    # 3. Current regime metadata
    cur_regime   = ust_reg["regime"].iloc[-1]
    cur_row      = all_streaks.iloc[-1]
    current_info = {
        "regime":    cur_regime,
        "start":     cur_row["start"],
        "duration":  cur_row["duration"],
        "last_date": ust_reg.index[-1],
        "matches":   cur_regime == regime_label,
    }

    # 4. Soberano metric series
    sob_series, sob_label = compute_sob_metric(sob_product, sob_type, sob_tenors)

    # 5. Filter streaks by label & date range
    mask = (
        (all_streaks["regime"] == regime_label) &
        (all_streaks["start"] >= pd.Timestamp(start_date)) &
        (all_streaks["start"] <= pd.Timestamp(end_date))
    )
    filtered = all_streaks[mask].copy()

    # 6. Build cumulative-change paths
    paths, path_dates = [], []
    for _, sk in filtered.iterrows():
        future = sob_series.index[sob_series.index >= sk["start"]][
            : EVENT_STUDY_HORIZON + 1
        ]
        if len(future) < 2:
            continue
        vals = sob_series.loc[future].values
        paths.append(vals - vals[0])
        path_dates.append(future.tolist())

    # 7. Current path (only if regime matches)
    current_path = None
    if current_info["matches"]:
        future = sob_series.index[sob_series.index >= current_info["start"]][
            : EVENT_STUDY_HORIZON + 1
        ]
        if len(future) >= 2:
            vals = sob_series.loc[future].values
            current_path = vals - vals[0]

    return {
        "paths":        paths,
        "path_dates":   path_dates,
        "current_path": current_path,
        "current_info": current_info,
        "streaks":      filtered,
        "sob_label":    sob_label,
    }


# ── CHART BUILDER ────────────────────────────────────────────────────────────

def build_event_study_chart(ust_spread, ust_lookback, regime_label,
                            sob_product, sob_type, sob_tenors,
                            start_date, end_date, band, show_mean):
    """Render the event-study Plotly chart."""

    res = extract_regime_paths(
        ust_spread, ust_lookback, regime_label,
        sob_product, sob_type, sob_tenors,
        start_date, end_date,
    )

    paths = res["paths"]
    ci    = res["current_info"]
    stk   = res["streaks"]
    label = res["sob_label"]

    if len(paths) == 0:
        print(f"No instances of '{regime_label}' found in the selected range.")
        return

    fig = go.Figure()

    # ── Historical paths (light gray) ────────────────────────────────────
    for i, p in enumerate(paths):
        x = list(range(len(p)))
        dates_i = res["path_dates"][i]
        hover = [
            f"Day {d}<br>{dates_i[d].strftime('%d %b %Y')}<br>"
            f"Cum Δ: {p[d]:+.1f} bps"
            for d in range(len(p))
        ]
        fig.add_trace(go.Scatter(
            x=x, y=p.tolist(),
            mode="lines",
            line=dict(color="rgba(180,180,180,0.6)", width=1),
            hovertext=hover, hoverinfo="text",
            showlegend=(i == 0),
            name="Historical paths" if i == 0 else "",
            legendgroup="hist",
        ))

    # ── Average path (yellow) & bands ────────────────────────────────────
    if show_mean and paths:
        max_len = min(max(len(p) for p in paths), EVENT_STUDY_HORIZON + 1)
        avg = np.full(max_len, np.nan)
        std = np.full(max_len, np.nan)
        pct_lo = np.full(max_len, np.nan)
        pct_hi = np.full(max_len, np.nan)

        for d in range(max_len):
            vals_d = [p[d] for p in paths if len(p) > d]
            if len(vals_d) >= 2:
                avg[d] = np.mean(vals_d)
                std[d] = np.std(vals_d, ddof=1)
                if band == "10th/90th":
                    pct_lo[d] = np.percentile(vals_d, 10)
                    pct_hi[d] = np.percentile(vals_d, 90)
                elif band == "25th/75th":
                    pct_lo[d] = np.percentile(vals_d, 25)
                    pct_hi[d] = np.percentile(vals_d, 75)
            elif len(vals_d) == 1:
                avg[d] = vals_d[0]
                std[d] = 0

        ok = ~np.isnan(avg)
        x_avg = np.arange(max_len)[ok]
        y_avg = avg[ok]
        y_std = std[ok]

        fig.add_trace(go.Scatter(
            x=x_avg.tolist(), y=y_avg.tolist(),
            mode="lines", line=dict(color="#FFD700", width=2.5),
            name="Mean path",
            hovertemplate="Day %{x}<br>Mean: %{y:+.1f} bps<extra></extra>",
        ))

        # Bands
        if band != "None" and len(x_avg) > 0:
            if band.endswith("SD"):
                n_sd = int(band[0])
                upper = y_avg + n_sd * y_std
                lower = y_avg - n_sd * y_std
                blabel = f"±{n_sd}σ"
            elif band in ("10th/90th", "25th/75th"):
                upper = pct_hi[ok]
                lower = pct_lo[ok]
                blabel = band.replace("/", "-") + " pctl"
            else:
                upper = lower = None
                blabel = ""

            if upper is not None:
                fig.add_trace(go.Scatter(
                    x=x_avg.tolist() + x_avg[::-1].tolist(),
                    y=upper.tolist() + lower[::-1].tolist(),
                    fill="toself",
                    fillcolor="rgba(255,215,0,0.10)",
                    line=dict(color="rgba(255,215,0,0.30)", width=1.5),
                    name=blabel, hoverinfo="skip",
                ))

    # ── Current path (red) ───────────────────────────────────────────────
    if res["current_path"] is not None:
        cp = res["current_path"]
        fig.add_trace(go.Scatter(
            x=list(range(len(cp))), y=cp.tolist(),
            mode="lines", line=dict(color="#FF4444", width=3),
            name="Current",
            hovertemplate="Day %{x}<br>Current: %{y:+.1f} bps<extra></extra>",
        ))

    # ── Zero line ────────────────────────────────────────────────────────
    fig.add_hline(y=0, line_dash="dash",
                  line_color="rgba(255,255,255,0.3)", line_width=0.5)

    # ── Info text ────────────────────────────────────────────────────────
    last_dt  = ci["last_date"].strftime("%Y-%m-%d")
    start_dt = ci["start"].strftime("%Y-%m-%d")
    match_s  = "(matches current)" if ci["matches"] else "(does NOT match current)"

    info1 = f"Current ({ust_spread}, {ust_lookback}d) as of {last_dt}: {ci['regime']}"
    info2 = f"In place for {ci['duration']} day(s) since {start_dt}"
    info3 = f"Selected: {regime_label} {match_s}"

    # Stats
    durs = stk["duration"].values
    stats = (
        f"Instances: {len(stk)} | "
        f"Longest: {durs.max()}d | Shortest: {durs.min()}d | "
        f"Range: {start_date} to {end_date}"
    )

    fig.update_layout(
        title=dict(
            text=(
                f"<b>Soberano Curve-Regime Event Study: {label}</b><br>"
                f"<span style='font-size:11px;color:#0051FF'>{info1}</span><br>"
                f"<span style='font-size:10px;color:#10AC10'>{info2}</span><br>"
                f"<span style='font-size:10px;color:#FFC400'>{info3}</span>"
            ),
            font=dict(color="Black", size=13),
        ),
        paper_bgcolor="#ffffff", plot_bgcolor="#ffffff",
        font=dict(color="Black"),
        yaxis=dict(
            title="Cumulative Δ (bps)", gridcolor="#ddd", gridwidth=0.5,
            zeroline=True, zerolinecolor="rgba(255,255,255,0.3)",
        ),
        xaxis=dict(
            title="Days in Regime", gridcolor="#ddd",
            dtick=5, range=[-0.5, EVENT_STUDY_HORIZON + 0.5],
        ),
        legend=dict(
            bgcolor="rgba(100,100,100,0)", bordercolor="#444",
            borderwidth=1, font=dict(size=9),
        ),
        height=550,
        margin=dict(t=120, b=50, l=60, r=20),
        annotations=[dict(
            text=stats, xref="paper", yref="paper",
            x=1, y=1.01, xanchor="right", yanchor="bottom",
            font=dict(color="#888", size=9), showarrow=False,
        )],
    )
    fig.show()


# ── WIDGET LAYOUT ────────────────────────────────────────────────────────────

w_es_product = widgets.Dropdown(
    options=["CMT Tenor", "LC Spread"], value="CMT Tenor",
    description="Product:", style={"description_width": "70px"},
    layout=widgets.Layout(width="220px"),
)
w_es_type = widgets.Dropdown(
    options=["Level", "Spread", "Butterfly"], value="Spread",
    description="Type:", style={"description_width": "50px"},
    layout=widgets.Layout(width="170px"),
)
w_es_t1 = widgets.Dropdown(
    options=SOB_TENOR_OPTIONS, value="6", description="T1:",
    style={"description_width": "30px"}, layout=widgets.Layout(width="100px"),
)
w_es_t2 = widgets.Dropdown(
    options=SOB_TENOR_OPTIONS, value="10", description="T2:",
    style={"description_width": "30px"}, layout=widgets.Layout(width="100px"),
)
w_es_t3 = widgets.Dropdown(
    options=SOB_TENOR_OPTIONS, value="15", description="T3:",
    style={"description_width": "30px"}, layout=widgets.Layout(width="100px"),
)

# Show/hide tenor boxes
w_es_t2.layout.display = "flex"
w_es_t3.layout.display = "none"

def _es_type_changed(change):
    w_es_t2.layout.display = "flex" if change["new"] in ("Spread", "Butterfly") else "none"
    w_es_t3.layout.display = "flex" if change["new"] == "Butterfly" else "none"
w_es_type.observe(_es_type_changed, names="value")

w_es_curve = widgets.Dropdown(
    options=list(UST_SPREAD_OPTIONS.keys()), value="2s10s",
    description="US Curve:", style={"description_width": "70px"},
    layout=widgets.Layout(width="180px"),
)
w_es_lookback = widgets.Dropdown(
    options=[("5d", 5), ("20d", 20)], value=20,
    description="Lookback:", style={"description_width": "70px"},
    layout=widgets.Layout(width="160px"),
)
w_es_regime = widgets.Dropdown(
    options=REGIME_ORDER, value="Flattener Twist",
    description="Regime:", style={"description_width": "60px"},
    layout=widgets.Layout(width="220px"),
)
w_es_band = widgets.Dropdown(
    options=["None", "1SD", "2SD", "3SD", "10th/90th", "25th/75th"],
    value="None",
    description="Band:", style={"description_width": "50px"},
    layout=widgets.Layout(width="160px"),
)
w_es_mean = widgets.Checkbox(
    value=True, description="Show mean",
    layout=widgets.Layout(width="120px"),
)

_es_default_start = pd.Timestamp("2016-01-01").date()
_es_default_end   = df_sob_cmt.index.max().date()

w_es_start = widgets.DatePicker(
    value=_es_default_start, description="Start:",
    style={"description_width": "50px"}, layout=widgets.Layout(width="200px"),
)
w_es_end = widgets.DatePicker(
    value=_es_default_end, description="End:",
    style={"description_width": "50px"}, layout=widgets.Layout(width="200px"),
)
w_es_btn = widgets.Button(
    description="Update", button_style="info",
    layout=widgets.Layout(width="90px"),
)

es_output = widgets.Output()

def _on_es_click(_):
    mt = w_es_type.value
    tenors = [int(w_es_t1.value)]
    if mt in ("Spread", "Butterfly"):
        tenors.append(int(w_es_t2.value))
    if mt == "Butterfly":
        tenors.append(int(w_es_t3.value))
    with es_output:
        clear_output(wait=True)
        build_event_study_chart(
            w_es_curve.value, w_es_lookback.value, w_es_regime.value,
            w_es_product.value, mt, tenors,
            str(w_es_start.value), str(w_es_end.value),
            w_es_band.value, w_es_mean.value,
        )

w_es_btn.on_click(_on_es_click)

display(widgets.VBox([
    widgets.HBox([w_es_product, w_es_type, w_es_t1, w_es_t2, w_es_t3]),
    widgets.HBox([w_es_curve, w_es_lookback, w_es_regime, w_es_band, w_es_mean]),
    widgets.HBox([w_es_start, w_es_end, w_es_btn]),
]))
display(es_output)
_on_es_click(None)

Output()

In [ ]:
# =============================================================================
# CELL 7 – PCA DECOMPOSITION OF LC SPREADS (Interactive)
# =============================================================================
# Runs PCA on the full LC Spread curve (LCS_6Y … LCS_15Y) and decomposes
# a selected target (single tenor or spread) into contributions from the
# first 6 principal components.
#   • Line chart  – evolution of each PC contribution over time
#   • Stacked bar – current-day decomposition (mean + 6 PCs + residual)
# =============================================================================
from sklearn.decomposition import PCA
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output

N_PCS = 6

# ── TARGET OPTIONS ───────────────────────────────────────────────────────────
# Single tenors
_pca_targets = {f"LCS_{t}Y": ("single", f"LCS_{t}Y") for t in range(6, 16)}
# Spreads (same pairs as Cell 4)
_pca_spread_pairs = {
    "LCS 6s10s": ("LCS_6Y", "LCS_10Y"),
    "LCS 6s15s": ("LCS_6Y", "LCS_15Y"),
    "LCS 7s15s": ("LCS_7Y", "LCS_15Y"),
    "LCS 6s12s": ("LCS_6Y", "LCS_12Y"),
    "LCS 7s10s": ("LCS_7Y", "LCS_10Y"),
    "LCS 8s12s": ("LCS_8Y", "LCS_12Y"),
    "LCS 9s15s": ("LCS_9Y", "LCS_15Y"),
    "LCS 10s15s": ("LCS_10Y", "LCS_15Y"),
}
for name, (s, l) in _pca_spread_pairs.items():
    _pca_targets[name] = ("spread", s, l)

PCA_TARGET_OPTIONS = list(_pca_targets.keys())

# ── COLORS ───────────────────────────────────────────────────────────────────
PC_COLORS = ["#4A90D9", "#D0021B", "#7ED321", "#F5A623", "#9B59B6", "#8B6914"]


def build_pca_charts(target_key, start_date, end_date):
    """Run PCA and plot decomposition for the selected target."""

    # Resolve target series
    cfg = _pca_targets[target_key]
    lcs_data = df_lcs.dropna()
    lcs_data = lcs_data[lcs_data.index >= pd.Timestamp(start_date)]
    lcs_data = lcs_data[lcs_data.index <= pd.Timestamp(end_date)]

    if len(lcs_data) < 30:
        print("Not enough data for PCA (need ≥30 dates).")
        return

    lcs_cols = sorted([c for c in lcs_data.columns if c.startswith("LCS_")],
                      key=lambda c: int(c.replace("LCS_", "").replace("Y", "")))

    if cfg[0] == "single":
        target_series = lcs_data[cfg[1]]
        target_label = cfg[1]
    else:  # spread
        target_series = lcs_data[cfg[2]] - lcs_data[cfg[1]]
        target_label = target_key

    X = lcs_data[lcs_cols].values
    X_mean = X.mean(axis=0)
    X_dm = X - X_mean

    # Fit PCA
    pca = PCA(n_components=N_PCS)
    scores = pca.fit_transform(X_dm)

    # For a single tenor, loading is direct; for a spread, it's long loading − short loading
    if cfg[0] == "single":
        col_idx = lcs_cols.index(cfg[1])
        loadings = pca.components_[:, col_idx]
    else:
        idx_s = lcs_cols.index(cfg[1])
        idx_l = lcs_cols.index(cfg[2])
        loadings = pca.components_[:, idx_l] - pca.components_[:, idx_s]

    # Contributions
    contributions = pd.DataFrame(index=lcs_data.index)
    for k in range(N_PCS):
        contributions[f"PC{k+1}"] = scores[:, k] * loadings[k]

    mean_target = target_series.mean()
    target_dm = target_series.values - mean_target
    residual = target_dm - contributions.values.sum(axis=1)

    print(f"PCA on {len(lcs_cols)} LC Spread tenors  |  {len(lcs_data):,} dates "
          f"({lcs_data.index.min().date()} to {lcs_data.index.max().date()})")
    print("Explained variance:  " +
          "  ".join(f"PC{k+1}: {pca.explained_variance_ratio_[k]*100:.1f}%"
                    for k in range(N_PCS)))
    print(f"Total ({N_PCS} PCs): {pca.explained_variance_ratio_.sum()*100:.1f}%")

    # ── CHART 1: Evolution line plot ─────────────────────────────────────────
    x_idx = list(range(len(contributions)))
    dates = contributions.index

    fig1 = go.Figure()
    for k in range(N_PCS):
        fig1.add_trace(go.Scatter(
            x=x_idx, y=contributions[f"PC{k+1}"].values,
            mode="lines",
            name=f"PC{k+1} ({pca.explained_variance_ratio_[k]*100:.1f}%)",
            line=dict(color=PC_COLORS[k], width=1.5),
            hovertemplate="%{customdata}<br>PC" + str(k+1) + ": %{y:+.1f} bps<extra></extra>",
            customdata=[d.strftime("%d %b %Y") for d in dates],
        ))

    n_ticks = min(15, len(x_idx))
    tick_pos = np.linspace(0, len(x_idx) - 1, n_ticks, dtype=int)
    last_dt = dates[-1].strftime("%Y-%m-%d")

    fig1.update_layout(
        title=dict(
            text=(
                f"<b>PCA Decomposition of {target_label}: Component Contributions</b><br>"
                f"<span style='font-size:11px;color:#555'>"
                f"Each line = PC score × loading  |  as of {last_dt}</span>"
            ),
            font=dict(color="black", size=13),
        ),
        paper_bgcolor="#ffffff", plot_bgcolor="#ffffff",
        font=dict(color="black"),
        yaxis=dict(title="Contribution (bps)", gridcolor="#ddd", zeroline=True,
                   zerolinecolor="rgba(0,0,0,0.15)"),
        xaxis=dict(
            tickvals=tick_pos.tolist(),
            ticktext=[dates[i].strftime("%b %y") for i in tick_pos],
            gridcolor="#ddd",
        ),
        legend=dict(
            bgcolor="rgba(255,255,255,0.9)", bordercolor="#ccc", borderwidth=1,
            font=dict(size=9), orientation="h",
            yanchor="top", y=-0.12, xanchor="center", x=0.5,
        ),
        dragmode="zoom", height=450,
        margin=dict(t=80, b=90, l=60, r=20),
    )
    fig1.show()

    # ── CHART 2: Current-day stacked bar ─────────────────────────────────────
    current_actual = target_series.iloc[-1]
    current_contribs = [contributions[f"PC{k+1}"].iloc[-1] for k in range(N_PCS)]
    current_resid = residual[-1]

    bar_labels = ["Mean"] + [f"PC{k+1}" for k in range(N_PCS)] + ["Residual"]
    bar_values = [mean_target] + current_contribs + [current_resid]
    bar_colors_list = ["#888888"] + PC_COLORS + ["#cccccc"]

    fig2 = go.Figure()
    cumulative = 0.0
    for i, (lbl, val, clr) in enumerate(zip(bar_labels, bar_values, bar_colors_list)):
        base = cumulative if i > 0 else 0
        fig2.add_trace(go.Bar(
            x=[target_label], y=[val], base=[base],
            name=f"{lbl}: {val:+.1f}",
            marker_color=clr,
            text=f"{val:+.1f}" if abs(val) > 0.5 else "",
            textposition="inside", textfont=dict(size=10),
            hovertemplate=f"{lbl}: {val:+.1f} bps<extra></extra>",
            width=0.5,
        ))
        cumulative += val

    fig2.add_trace(go.Scatter(
        x=[target_label], y=[current_actual],
        mode="markers+text",
        marker=dict(color="#D0021B", size=12, symbol="diamond"),
        text=[f"  Total: {current_actual:.1f}"],
        textposition="middle right", textfont=dict(size=11, color="#D0021B"),
        name=f"Actual: {current_actual:.1f}",
        hovertemplate=f"Actual: {current_actual:.1f} bps<extra></extra>",
    ))

    fig2.update_layout(
        title=dict(
            text=(
                f"<b>{target_label} Current Decomposition</b><br>"
                f"<span style='font-size:11px;color:#555'>"
                f"As of {last_dt}  |  Actual: {current_actual:.1f} bps  |  "
                f"Mean: {mean_target:.1f} bps</span>"
            ),
            font=dict(color="black", size=13),
        ),
        paper_bgcolor="#ffffff", plot_bgcolor="#ffffff",
        font=dict(color="black"),
        yaxis=dict(title="bps", gridcolor="#ddd", zeroline=True,
                   zerolinecolor="rgba(0,0,0,0.15)"),
        xaxis=dict(showticklabels=False),
        legend=dict(
            bgcolor="rgba(255,255,255,0.9)", bordercolor="#ccc", borderwidth=1,
            font=dict(size=9),
        ),
        barmode="stack", height=450,
        margin=dict(t=80, b=40, l=60, r=120),
    )
    fig2.show()

    print(f"\nCurrent {target_label} = {current_actual:.1f} bps")
    print(f"  Mean:     {mean_target:+.1f}")
    for k in range(N_PCS):
        print(f"  PC{k+1}:     {current_contribs[k]:+.1f}")
    print(f"  Residual: {current_resid:+.1f}")
    print(f"  ─────────────────")
    print(f"  Sum:      {mean_target + sum(current_contribs) + current_resid:.1f}")


# ── WIDGET LAYOUT ────────────────────────────────────────────────────────────

w_pca_target = widgets.Dropdown(
    options=PCA_TARGET_OPTIONS, value="LCS_10Y",
    description="Target:", style={"description_width": "60px"},
    layout=widgets.Layout(width="220px"),
)

_pca_default_start = pd.Timestamp("2013-01-01").date()
_pca_default_end = df_lcs.index.max().date()

w_pca_start = widgets.DatePicker(
    value=_pca_default_start, description="Start:",
    style={"description_width": "50px"}, layout=widgets.Layout(width="200px"),
)
w_pca_end = widgets.DatePicker(
    value=_pca_default_end, description="End:",
    style={"description_width": "50px"}, layout=widgets.Layout(width="200px"),
)
w_pca_btn = widgets.Button(
    description="Update", button_style="info",
    layout=widgets.Layout(width="90px"),
)

pca_output = widgets.Output()

def _on_pca_click(_):
    with pca_output:
        clear_output(wait=True)
        build_pca_charts(
            w_pca_target.value,
            str(w_pca_start.value), str(w_pca_end.value),
        )

w_pca_btn.on_click(_on_pca_click)

display(widgets.HBox([w_pca_target, w_pca_start, w_pca_end, w_pca_btn]))
display(pca_output)
_on_pca_click(None)